## Identify missing oligos and plot them per label
### Outline:
1. Find number of oligos and their names `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesign/reference.fa`
2. Find all assigned oligos with the barcodes
3. Take the difference
4. How many missings per group?

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import gzip

In [2]:
! pwd

/data/gpfs-1/work/users/kisa11_c/coding/80K_analysis/01_missing_sequences


In [ ]:
def make_header_list(fasta_file):
    header_list = []
    with open(fasta_file, 'r') as fh:
        for line in fh:
            if line.startswith('>'):
                header_list.append(line.strip().lstrip('>'))
    # make df from list
    header_df = pd.DataFrame(header_list)
    return header_df # or use cat reference.fa | grep ">" | awk '{print substr($0,2)}' > all_headers.tsv

In [2]:
all_headers = '/fast/work/groups/ag_kircher/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesign/all_ref_sequences.tsv'
all_headers = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/reference/all_headers.tsv'
# read in all sequences as tsv
all_seq_df = pd.read_csv(all_headers, header=None, sep='\t')
all_seq_df.columns = ['oligo_name']
# print(all_seq_df.shape) # 80214
all_seq_df

,oligo_name
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....
...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...
80212,MK:tile_18415|chr17-71181691+71181960|scramble...
80213,MK:tile_14356|chr15-67031618+67031887|scramble...


In [3]:
# add second column which is split at first : and contains the label
all_seq_df['label'] = all_seq_df['oligo_name'].str.split(':').str[0]


In [5]:
all_seq_df
# show all unique labels
print(all_seq_df['label'].unique())
# print(len(all_seq_df['label'].unique())) # 29

['cardiac_neuro_cava_random' 'GC_Atrial_fib' 'GC_Liang' 'GC_Selvarajan'
 'GC_Mohlke' 'GC_Kircher' 'GC_Mendelian_variants' 'C_positive_heart_CAD'
 'GC_Cort_Chengyu' 'GC_GABA_Chengyu' 'GC_Glut_Chengyu' 'GC_Hon' 'GC_Vista'
 'GC_DNase_positive' 'GC_DNase_negative_brain' 'GC_DNase_negative_blood'
 'C_negative_heart_MK' 'C_negative_neuron_MK' 'C_negative_neuron_NP'
 'C_positive_heart_MK' 'C_positive_neuron_CD' 'C_positive_neuron_MK'
 'C_positive_neuron_NP' 'C_positive_heart_AB' 'C_SLEA'
 'GC_DNase_positive_shuffeled' 'GC_DNase_negative_brain_shuffeled'
 'GC_DNase_negative_blood_shuffeled' 'MK']


## Read all assigned oligos with the barcodes

In [ ]:
# remove the third column of assignment_barcodes.standardConfig.sorted.tsv.gz
# zcat assignment_barcodes.standardConfig.sorted.tsv.gz | head -n 10 | awk '{print $1, $2, $4}' | sort | uniq -c | sort -nr > assignment_barcodes.standardConfig_

In [95]:
# all assigned sequences with barcode
assigned_seq = '/fast/work/groups/ag_kircher/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesign/assignment_barcodes.standardConfig.sorted.tsv.gz'
# assigned_seq = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/assignment_barcodes.standardConfig.sorted.tsv.gz'
# assigned_seq = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/standard_results/results/assignment/standardAssignIGVFDesignNoTemp/assignment_barcodes.standardConfig.sorted.tsv.gz'
assigned_seq_df = pd.read_csv(assigned_seq, sep='\t', header=None)
assigned_seq_df.columns = ['barcode', 'oligo_name', 'quality', 'number of matches']
assigned_seq_df # 7078311 rows × 4 columns


,barcode,oligo_name,quality,number of matches
0,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60,5/5
1,AAAAAAAAAACAAGT,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,7/7
2,AAAAAAAAAACACCA,cardiac_neuro_cava_random:ALT_FTO|ENSG00000140...,16;270M;NM:i:0;MD:Z:270;6,10/10
3,AAAAAAAAAACCTCG,cardiac_neuro_cava_random:REF_FKRP|ENSG0000018...,16;270M;NM:i:0;MD:Z:270;6,7/7
4,AAAAAAAAAAGCTGG,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,8/8
...,...,...,...,...
7078306,TTTTTTTTTTGACGA,cardiac_neuro_cava_random:ALT_ACTN2|ENSG000000...,16;270M;NM:i:0;MD:Z:270;6,11/11
7078307,TTTTTTTTTTGCACA,cardiac_neuro_cava_random:ALT_CARD11|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,25/26
7078308,TTTTTTTTTTGCACC,cardiac_neuro_cava_random:ALT_KRAS|ENSG0000013...,16;214M1I56M;NM:i:1;MD:Z:270;5,6/7
7078309,TTTTTTTTTTGCTAA,cardiac_neuro_cava_random:REF_NR4A2|ENSG000001...,16;270M;NM:i:0;MD:Z:270;6,6/6


In [96]:
# How many barcodes and sequqnces are duplicated?
barcode_counts = assigned_seq_df['barcode'].value_counts()
assigned_oligo_counts = assigned_seq_df['oligo_name'].value_counts()

In [97]:
# check if there are barcodes with more than one assigned sequence
print(barcode_counts[barcode_counts > 1]) # empty array -> no barcode is assigned to more than one sequence

# sequences: 
# len(assigned_oligo_counts[assigned_oligo_counts > 1]) # 74594
print(assigned_oligo_counts[assigned_oligo_counts > 1]) # assigned_oligo_counts (len: 75131) sequences with more than 1 barcode assigned: 74594

Series([], Name: count, dtype: int64)
oligo_name
GC_GABA_Chengyu:GABA|chr10:26798829-26799098|-|1.62                                                                                      1066
MK:tile_47615|chr14-29242851+29243121|reference                                                                                           893
GC_GABA_Chengyu:GABA|chr10:26798799-26799068|-|1.71                                                                                       860
MK:tile_47607|chr13-80235477+80235747|reference                                                                                           853
MK:tile_47617|chr14-29242931+29243201|reference                                                                                           848
                                                                                                                                         ... 
cardiac_neuro_cava_random:ALT_CNOT3|ENSG00000088038.20|EH38E3316446_fwd_tile1-1_CNOT3|ENSG000000880

In [9]:
# left join from all_seq_df to assigned_seq_df on oligo_name (wrong direction for checking which oligos are only in reference)
merged_df = pd.merge(assigned_seq_df, all_seq_df, on='oligo_name', how='left')

In [10]:
# how many rows have no label?
merged_df

,barcode,oligo_name,quality,number of matches,label
0,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,16;270M;NM:i:0;MD:Z:270;60,5/5,GC_Vista
1,AAAAAAAAAACAAGT,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,7/7,cardiac_neuro_cava_random
2,AAAAAAAAAACACCA,cardiac_neuro_cava_random:ALT_FTO|ENSG00000140...,16;270M;NM:i:0;MD:Z:270;6,10/10,cardiac_neuro_cava_random
3,AAAAAAAAAACCTCG,cardiac_neuro_cava_random:REF_FKRP|ENSG0000018...,16;270M;NM:i:0;MD:Z:270;6,7/7,cardiac_neuro_cava_random
4,AAAAAAAAAAGCTGG,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,8/8,cardiac_neuro_cava_random
...,...,...,...,...,...
7078306,TTTTTTTTTTGACGA,cardiac_neuro_cava_random:ALT_ACTN2|ENSG000000...,16;270M;NM:i:0;MD:Z:270;6,11/11,cardiac_neuro_cava_random
7078307,TTTTTTTTTTGCACA,cardiac_neuro_cava_random:ALT_CARD11|ENSG00000...,16;270M;NM:i:0;MD:Z:270;6,25/26,cardiac_neuro_cava_random
7078308,TTTTTTTTTTGCACC,cardiac_neuro_cava_random:ALT_KRAS|ENSG0000013...,16;214M1I56M;NM:i:1;MD:Z:270;5,6/7,cardiac_neuro_cava_random
7078309,TTTTTTTTTTGCTAA,cardiac_neuro_cava_random:REF_NR4A2|ENSG000001...,16;270M;NM:i:0;MD:Z:270;6,6/6,cardiac_neuro_cava_random


In [98]:
# left join on reference by oligo_name (to check which oligos are only in reference)
all_seq_with_assignment = all_seq_df.merge(assigned_seq_df, on='oligo_name', how='left') # bwa: 7083395 
all_seq_with_assignment

,oligo_name,label,barcode,quality,number of matches
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACATCATGGCTGAC,16;270M;NM:i:0;MD:Z:270;60,8/8
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACCCACAATACAGG,16;270M;NM:i:0;MD:Z:270;60,9/9
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACTTCAGCAAACCG,16;270M;NM:i:0;MD:Z:270;60,26/26
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AATTTACAGCATTCG,16;270M;NM:i:0;MD:Z:270;60,3/3
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,ACACGCTGAATATAG,16;270M;NM:i:0;MD:Z:270;60,3/3
...,...,...,...,...,...
7083390,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTAGGTTCAACCCGG,16;270M;NM:i:12;MD:Z:42A2C2T24C1T7T8C4G12C0T2A...,9/9
7083391,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTATAGCATACCTAC,16;270M;NM:i:15;MD:Z:41T0A5T7C6C9C6T3A3T16G1T2...,7/7
7083392,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTCTAAGCTTGACGC,16;270M;NM:i:10;MD:Z:41T6T8T5C7T1C6T2T16T10T15...,6/6
7083393,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTTAAATCCTTGATA,16;270M;NM:i:11;MD:Z:42A13C16C3C2T24G0C3C0T2A4...,6/6


In [40]:
# how many rows of all_seq_with_assignment have no label
print(len(all_seq_with_assignment[all_seq_with_assignment['barcode'].isna()])) # 5084 rows

# get only these sequence names and store them in a tsv file (name and label)
missing_sequences = all_seq_with_assignment[all_seq_with_assignment['barcode'].isna()][['oligo_name', 'label']]
# missing_sequences["oligo_name"].unique().shape # all are unique
# ! go through df and count which sequences are missing 

5084


In [104]:
bwa_missing_seqs = all_seq_with_assignment[all_seq_with_assignment['barcode'].isna()]
bwa_missing_seqs

,oligo_name,label,barcode,quality,number of matches
1725,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,cardiac_neuro_cava_random,NaN,NaN,NaN
3539,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,cardiac_neuro_cava_random,NaN,NaN,NaN
8201,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,cardiac_neuro_cava_random,NaN,NaN,NaN
13209,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,cardiac_neuro_cava_random,NaN,NaN,NaN
31540,cardiac_neuro_cava_random:RERE|ENSG00000142599...,cardiac_neuro_cava_random,NaN,NaN,NaN
...,...,...,...,...,...
7028722,MK:tile_985|chr1-33363966+33364235|LC28t6,MK,NaN,NaN,NaN
7028723,MK:tile_985|chr1-33363966+33364235|LC28t7,MK,NaN,NaN,NaN
7028725,MK:tile_985|chr1-33363966+33364235|LC28t9,MK,NaN,NaN,NaN
7068818,MK:tile_30307|chr3-171305106+171305375|scrambl...,MK,NaN,NaN,NaN


In [41]:
all_seq_with_assignment

,oligo_name,label,barcode,quality,number of matches
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACATCATGGCTGAC,16;270M;NM:i:0;MD:Z:270;60,8/8
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACCCACAATACAGG,16;270M;NM:i:0;MD:Z:270;60,9/9
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AACTTCAGCAAACCG,16;270M;NM:i:0;MD:Z:270;60,26/26
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,AATTTACAGCATTCG,16;270M;NM:i:0;MD:Z:270;60,3/3
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random,ACACGCTGAATATAG,16;270M;NM:i:0;MD:Z:270;60,3/3
...,...,...,...,...,...
7083390,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTAGGTTCAACCCGG,16;270M;NM:i:12;MD:Z:42A2C2T24C1T7T8C4G12C0T2A...,9/9
7083391,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTATAGCATACCTAC,16;270M;NM:i:15;MD:Z:41T0A5T7C6C9C6T3A3T16G1T2...,7/7
7083392,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTCTAAGCTTGACGC,16;270M;NM:i:10;MD:Z:41T6T8T5C7T1C6T2T16T10T15...,6/6
7083393,MK:tile_14356|chr15-67031618+67031887|scramble...,MK,TTTAAATCCTTGATA,16;270M;NM:i:11;MD:Z:42A13C16C3C2T24G0C3C0T2A4...,6/6


In [42]:
# return a list of all oligo_name's where barcode is NaN
missing_seqs = all_seq_with_assignment.loc[all_seq_with_assignment['barcode'].isna()] # 5084

In [43]:
# drop all columns except oligo_name and label
missing_seqs = missing_seqs.drop(['barcode', 'quality', 'number of matches'], axis=1)


In [44]:
# label groups with missing sequences:
label_groups_w_missing = missing_seqs['label'].unique()
len(label_groups_w_missing)
# label groups without missing sequences: find all labels that are not in label_groups_w_missing
label_groups_no_missing = [x for x in all_seq_df['label'].unique() if x not in label_groups_w_missing]
print(len(label_groups_no_missing)) # 8
label_groups_no_missing
# ['GC_Liang',
#  'GC_Mohlke',
#  'C_positive_heart_CAD',
#  'GC_Cort_Chengyu',
#  'GC_DNase_positive',
#  'GC_DNase_negative_brain',
#  'GC_DNase_negative_blood',
#  'GC_DNase_negative_brain_shuffeled']


8


['GC_Liang',
 'GC_Mohlke',
 'C_positive_heart_CAD',
 'GC_Cort_Chengyu',
 'GC_DNase_positive',
 'GC_DNase_negative_brain',
 'GC_DNase_negative_blood',
 'GC_DNase_negative_brain_shuffeled']

In [130]:
# plot the counts per label

# # check for duplicated oligo_names in missing_seqs
# missing_seqs['oligo_name'].value_counts() # all are unique
sorted_missing_seqs = missing_seqs.groupby('label').count().sort_values(by='oligo_name', ascending=False)
ax = sorted_missing_seqs.plot(kind='bar', figsize=(20, 6))
for p in ax.patches:
    ax.annotate(str(p.get_height()), (p.get_x() * 1.005, (p.get_height() + 15) * 1.005))

oligo_name
cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779764_fwd_tile1-1                                                                                                                   1
cardiac_neuro_cava_random:ALT_TCAP|ENSG00000173991.6|EH38E1860996_fwd_tile1-1_NEUROD2|ENSG00000171532.5|EH38E1860996|17-39643319-C-T~TCAP|ENSG00000173991.6|EH38E1860996|17-39643319-C-T       1
cardiac_neuro_cava_random:ALT_NEUROD2|ENSG00000171532.5|EH38E3221677_rev_tile1-1_NEUROD2|ENSG00000171532.5|EH38E3221677|17-39645162-T-C~TCAP|ENSG00000173991.6|EH38E3221677|17-39645162-T-C    1
cardiac_neuro_cava_random:ALT_TCAP|ENSG00000173991.6|EH38E3221677_fwd_tile1-1_NEUROD2|ENSG00000171532.5|EH38E3221677|17-39645162-T-C~TCAP|ENSG00000173991.6|EH38E3221677|17-39645162-T-C       1
cardiac_neuro_cava_random:ALT_NEUROD2|ENSG00000171532.5|EH38E1860996_rev_tile1-1_NEUROD2|ENSG00000171532.5|EH38E1860996|17-39643468-G-A~TCAP|ENSG00000173991.6|EH38E1860996|17-39643468-G-A    1
                        

## Which sequences are this exactly?
- Perpare list of unmapped sequenes
- Take two sequences by random and match them to the reference
- Write function which produces fasta file for the unmapped sequences
  - Iterate over reference.fa and take only the list of the unmapped sequences

In [22]:
# missing_seqs['oligo_name']
import yaml
# load reference fasta specified in config
config_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/config.yaml'
assignment_name = 'assignIGVFDesignNoTemp'

with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()
ref_fasta_path = config['assignments'][assignment_name]['reference']




In [23]:
len(missing_seqs['oligo_name'].values.tolist()) # 5084

5084

In [24]:
# subsampled data (400 lines)
# ref_fasta_path = '/data/gpfs-1/users/kisa11_c/work/coding/tmp_data/tmp_ref_no_dup.fa'
missing_seqs_list = missing_seqs['oligo_name'].values.tolist()
missing_seq_dict = prepare_missing_seq_dict(ref_fasta_path, missing_seqs_list)

NameError: name 'prepare_missing_seq_dict' is not defined

In [ ]:
len(missing_seq_dict)
missing_seq_dict.values()

{'>cardiac_neuro_cava_random:PRDM16|ENSG00000142611.17|EH38E2779764_fwd_tile1-1': 'AGGACCGGATCAACTGCCGCCCAGGAGCTCTCGTGCATCCACTCTGGTCCTCCGGTCCCGGCTGCGCCTCTTGCACCAGGCTGGGGCAGGGATTACCAGCCGCACGCAGGCTGCGGGAACCCCCTTTGTCTGGCTTTCGGCGGAGTCGGCAGAGTTCCTTCCTTCTGGGCTAATGCCCAGTTTAATTGTACATCCCATTGTGTCGTCTCTGTTCAATCATGTTCAAAAATACCTACGTCCACTCCGTTCCCATTTAGATCTCTCTAAAGTCCATTCCGGCTTATCCATTGCGTGAACCGA',
 '>cardiac_neuro_cava_random:SZT2|ENSG00000198198.17|EH38E2807306_fwd_tile1-1': 'AGGACCGGATCAACTGGGGGCGTGTGGTGGGTGGGGGGTGGGTGTTGCTAATTTAGACTGAGTGGCCAGGAAAAGCCTCACCAGGGAGGTGACAGATAAGCCGAGATCTAAATGGCAAGAAGGAATGAGTCACATGAAGACCTACAGTCAGAGCAATCCAGGGCAAAGGCAAAGTGCAAAGATAGCACATTTGGCATATCTATGGCACAGAAAGAAGGCCTGTGCGGATGAAGGGTGATAAACTGGCATGAGAATCATAAGAGATGAGAATGGCAAGGCAAGGAGCATTGCGTGAACCGA',
 '>cardiac_neuro_cava_random:ST3GAL3|ENSG00000126091.21|EH38E2807766_fwd_tile1-1': 'AGGACCGGATCAACTCCCCAACCTCTCTCACGTACACCTGCGTGTTCATGTACACATACATGTACATAGCATCCCTTGAGCTGTCCTCACCTGACCTACTGAACTCTGAGGTGGTCCGGGTCCCCCAGAGATGGCTGGGTAGCTGG

In [ ]:
## fastq files
# /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTemp/fastq/merge_split0.join.fastq.gz

merged_reads = ["merge_split0.join.fastq.gz", "merge_split13.join.fastq.gz", "merge_split18.join.fastq.gz", "merge_split22.join.fastq.gz", "merge_split27.join.fastq.gz", "merge_split5.join.fastq.gz", "merge_split1.join.fastq.gz", "merge_split14.join.fastq.gz", "merge_split19.join.fastq.gz", "merge_split23.join.fastq.gz", "merge_split28.join.fastq.gz", "merge_split6.join.fastq.gz", "merge_split10.join.fastq.gz", "merge_split15.join.fastq.gz", "merge_split2.join.fastq.gz", "merge_split24.join.fastq.gz", "merge_split29.join.fastq.gz", "merge_split7.join.fastq.gz", "merge_split11.join.fastq.gz", "merge_split16.join.fastq.gz", "merge_split20.join.fastq.gz", "merge_split25.join.fastq.gz", "merge_split3.join.fastq.gz", "merge_split8.join.fastq.gz", "merge_split12.join.fastq.gz", "merge_split17.join.fastq.gz", "merge_split21.join.fastq.gz", "merge_split26.join.fastq.gz", "merge_split4.join.fastq.gz", "merge_split9.join.fastq.gz"]

In [ ]:
def prep_query_seq(seq, length, reverse=False):
    '''
    Prepares the query sequence for the search
    @param: sequence with full length; length: disired length; reverse: bool if reverse sequence is required
    @output: query sequence
    '''
    if reverse:
        query_seq = seq[::-1]
    return seq[:length]
        
# feature assignment exact matches 
def exact_match(fastq_file_path, seq_dict, seq_length, reverse=False):
    '''Tries to find all sequences in one read sequence. Iterates each missing sequence for an exact match. The sequence length is given by the user.'''
    seperation = ''.join(['-'] * 20)
    counter = 0
    with gzip.open(fastq_file_path, 'rt') as reads_fastq:
        with open(_output_file_path, 'w') as output_file:
            for line in reads_fastq:
                if line.startswith('@'):
                    header = line
                elif line.startswith('+'):
                    pass
                else:
                    read_seq = line
                    for header, seq in seq_dict.items():
                        query_seq = prep_query_seq(seq, seq_length, reverse)
                        if query_seq in read_seq:
                            counter += 1
                            output_file.write(f'{seperation}\n\nFound match\n')
                            output_file.write(f'In header: {header}\n')
                            output_file.write(f'match of length {seq_length} of\nquery: {header}\n\nseq:{query_seq}\n\n')
                            output_file.write(f'in sequence: {read_seq}\n')
                            output_file.write(f'{seperation}\n')
                            # print(f'{seperation}\n\nFound match\n')
                            # print('In header: ',header)
                            # print(f'match of length {seq_length} of ', query_seq)
                            # print('in sequence: ', seq)
                            # print(f'{seperation}')
            output_file.write(f'Found {counter} matches in {fastq_file_path}\n')
    print(f'Found {counter} matches')
# main

In [ ]:
    test_search_dict = {
        'search header': 'TCCCAGGTGAGATGGGGAGGTGAGTAGCAGATGATCTCGTGGAAGCTCCTCACCAACCTCCATCCTCTCAGTTCCTGGAAGATCACAGGGTGTTCTGTGAAGACTCAGAT'
    }
    # test_path = '/home/kisa/coding/scripts/test.fastq'
    merged_reads_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTemp/fastq/merge_split0.join.fastq.gz'
    exact_match(merged_reads_path, missing_seq_dict, 110, False)

NameError: name 'exact_match' is not defined

In [ ]:
import json 
json.dump(missing_seq_dict, open('/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/missing_seq_dict.json', 'w+'))

## Filter reference_exact for missing sequences


In [ ]:
# write missing sequences to file (.fa)
import json 
missing_seq_dict = json.load(open('/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/missing_seq_dict.json', 'r'))

In [ ]:
with open('/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/all_missing_sequences.fa', 'w') as missing_seq_file:
    for header, seq in missing_seq_dict.items():
        missing_seq_file.write(f'{header}\n{seq}\n')
    

## Script:

In [ ]:
# import:
import pandas as pd

## input:
assigned_barcodes = '/fast/work/groups/ag_kircher/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesign/assignment_barcodes.standardConfig.sorted.tsv.gz'

# all assigned sequences with barcode
assigned_seq_df = pd.read_csv(assigned_barcodes, sep='\t', header=None)

### Check bowtie results samtools idxstats from merged bam files
- "/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie.tsv"

In [56]:
bowtie_results = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie.tsv'
bowtie_results = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie2.tsv'
bowtie_df = pd.read_csv(bowtie_results, sep='\t', header=None)
bowtie_df.columns = ['oligo_name', 'length', 'mapped_reads', 'unmapped_reads']

In [57]:
bowtie_df

,oligo_name,length,mapped_reads,unmapped_reads
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,636,0
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,441,0
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,206,0
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,164,0
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,717,0
...,...,...,...,...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,300,88,0
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,300,2647,0
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,300,431,0
80214,MK:tile_22033|chr2-71506006+71506275|scramble_...,300,0,0


In [58]:
# add label as column
bowtie_df["label"] = bowtie_df["oligo_name"].str.split(':').str[0] # 80215 + 1 
# lost because no/not enough reads found
missing_bowtie = bowtie_df[bowtie_df['mapped_reads'] < 10] # bowtie: 2635 +1 bowie2: 2110 + 1 
# remove the sequences without mapped reads
bowtie_df = bowtie_df[bowtie_df['mapped_reads'] > 9] # bowtie: >0: 78613; >9: 77580 bowtie2: >0 79092; >9 78105
bowtie_df

,oligo_name,length,mapped_reads,unmapped_reads,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,636,0,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,441,0,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,206,0,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,164,0,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,717,0,cardiac_neuro_cava_random
...,...,...,...,...,...
80209,MK:tile_24954|chr2-236620899+236621168|scrambl...,300,152,0,MK
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,300,2668,0,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,300,88,0,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,300,2647,0,MK


In [59]:
missing_bowtie

,oligo_name,length,mapped_reads,unmapped_reads,label
21,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,9,0,cardiac_neuro_cava_random
50,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,300,7,0,cardiac_neuro_cava_random
93,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,300,1,0,cardiac_neuro_cava_random
149,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,300,5,0,cardiac_neuro_cava_random
208,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,300,8,0,cardiac_neuro_cava_random
...,...,...,...,...,...
79706,MK:tile_985|chr1-33363966+33364235|LC28t7,300,3,0,MK
79708,MK:tile_985|chr1-33363966+33364235|LC28t9,300,7,0,MK
80066,MK:tile_30307|chr3-171305106+171305375|scrambl...,300,0,0,MK
80214,MK:tile_22033|chr2-71506006+71506275|scramble_...,300,0,0,MK


In [60]:
# find the sequences that are not in the bowtie_df
missing_bowtie # 2635 + 1 (one line with * as oligo_name => filter out)
# filter out the line with *
missing_bowtie = missing_bowtie[missing_bowtie['oligo_name'] != '*'] 
# print the value count distribution of mapped_reads
missing_bowtie['mapped_reads'].value_counts() # bowtie: 9: 55, 8: 73, 7: 64 (sum: 2635) bowtie2: 9: 60, 8: 64, 7: 92 (sum: 2110)

mapped_reads
0    1123
1     254
2     158
3     120
4     100
7      92
5      80
8      64
9      60
6      59
Name: count, dtype: int64

In [61]:
missing_bowtie

,oligo_name,length,mapped_reads,unmapped_reads,label
21,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,9,0,cardiac_neuro_cava_random
50,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,300,7,0,cardiac_neuro_cava_random
93,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,300,1,0,cardiac_neuro_cava_random
149,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,300,5,0,cardiac_neuro_cava_random
208,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,300,8,0,cardiac_neuro_cava_random
...,...,...,...,...,...
79705,MK:tile_985|chr1-33363966+33364235|LC28t6,300,4,0,MK
79706,MK:tile_985|chr1-33363966+33364235|LC28t7,300,3,0,MK
79708,MK:tile_985|chr1-33363966+33364235|LC28t9,300,7,0,MK
80066,MK:tile_30307|chr3-171305106+171305375|scrambl...,300,0,0,MK


In [133]:
mappend_thres = 9
missing_thres = 10


def plot_label_distribution(bowtie_results, mapped_threshold, missing_threshold=10):
    """
    Plot the distribution of labels in a df (samtools idxstats result)
    @param: bowtie_results: path to bowtie results file; mapped_threshold: threshold for mapped reads
    """
    bowtie_df = pd.read_csv(bowtie_results, sep='\t', header=None)
    bowtie_df.columns = ['oligo_name', 'length', 'mapped_reads', 'unmapped_reads']

    # add label as column
    bowtie_df["label"] = bowtie_df["oligo_name"].str.split(':').str[0]
    # lost because no/not enough reads found
    missing_bowtie = bowtie_df[bowtie_df['mapped_reads'] < missing_threshold]
    # filter out the line with *
    missing_bowtie = missing_bowtie[missing_bowtie['oligo_name'] != '*'] 

    # remove the sequences without mapped reads
    bowtie_df = bowtie_df[bowtie_df['mapped_reads'] > mapped_threshold] 

    # print number of missing sequences
    print(f"\n---------Summary----------\nNumber of missing sequences with less than {missing_threshold} is {len(missing_bowtie)} while {len(bowtie_df)} sequences have more than {mapped_threshold} mapped reads.") 

    # # plot the distribution of labels
    # sorted_missing_seqs_bowtie = missing_bowtie.groupby('label').count().sort_values(by='oligo_name', ascending=False)
    # # drop columns that are not needed (all except oligo_name and label)
    # sorted_missing_seqs_bowtie = sorted_missing_seqs_bowtie.drop(['length', 'mapped_reads', 'unmapped_reads'], axis=1)
    # ax = sorted_missing_seqs_bowtie.plot(kind='bar', figsize=(20, 6))
    # for p in ax.patches:
    #     ax.annotate(str(p.get_height()), (p.get_x() * 1.005, (p.get_height() + 15) * 1.005))
    return bowtie_df, missing_bowtie

In [134]:
bowtie_results = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie.tsv'
bowtie2_results = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/experiment/results/assignment/assignIGVFDesignNoTempBowtie/bam/idxstats_bowtie2.tsv'

bowtie_df, missing_bowtie = plot_label_distribution(bowtie_results, mappend_thres, missing_thres)
bowtie2_df, missing_bowtie2 = plot_label_distribution(bowtie2_results, mappend_thres, missing_thres)


---------Summary----------
Number of missing sequences with less than 10 is 2635 while 77580 sequences have more than 9 mapped reads.
2635
2635

---------Summary----------
Number of missing sequences with less than 10 is 2110 while 78105 sequences have more than 9 mapped reads.
2110
2110


#### Solution using functions

In [ ]:
# plot the distribution of labels
sorted_missing_seqs_bowtie = missing_bowtie.groupby('label').count().sort_values(by='oligo_name', ascending=False)
# drop columns that are not needed (all except oligo_name and label)
sorted_missing_seqs_bowtie = sorted_missing_seqs_bowtie.drop(['length', 'mapped_reads', 'unmapped_reads'], axis=1)
ax = sorted_missing_seqs_bowtie.plot(kind='bar', figsize=(20, 6))
for p in ax.patches:
    ax.annotate(str(p.get_height()), (p.get_x() * 1.005, (p.get_height() + 15) * 1.005))

### Identify if the aligners have missing sequences with exact matches
- read all all_exact_match.tsv: /data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/all_exact_match.tsv
- find if there is a sequence which is missing by an aligner but in the exact matches 

In [92]:
# read exact matches 
exact_matches = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/01_missing_sequences/results/match_missing_sequences/all_exact_match.tsv'
exact_matches_df = pd.read_csv(exact_matches, sep='\t', header=None)
# add column names
exact_matches_df.columns = ['read', 'header', 'seq', 'match_length']
exact_matches_df




,read,header,seq,match_length
0,@NB501960:812:HH53WAFX5:1:11101:21622:1110 XI:...,>cardiac_neuro_cava_random:REF_PPP2R2A|ENSG000...,ACTGCAATGCTGCTAATGGATTTTGAGTTGTGGGTACCATATACCC...,270M
1,@NB501960:812:HH53WAFX5:1:11101:4019:1255 XI:Z...,>cardiac_neuro_cava_random:REF_ANK2|ENSG000001...,GTTTTTAGTTGTGTTTAAATCAGGAATTTCCTTACTTTTCAAAATT...,270M
2,@NB501960:812:HH53WAFX5:1:11101:3848:1256 XI:Z...,>cardiac_neuro_cava_random:REF_LMNA|ENSG000001...,TCTTAAATTATCTGAATCTCTTCTGAAGACAGACCTATTAGCTTTT...,270M
3,@NB501960:812:HH53WAFX5:1:11101:5659:1395 XI:Z...,>MK:newcore_110746|chr13-80235318+80235587|ref...,GCTTCTAGTCATAGACAATGGCAATTGCAGAGCAAGACTAATAACA...,270M
4,@NB501960:812:HH53WAFX5:1:11101:23682:1535 XI:...,>cardiac_neuro_cava_random:ALT_RAI1|ENSG000001...,GAGCAACGCAGGAGAGCAGAAAGGCGAGTGCCCTGGGCTCAGGGGG...,270M
...,...,...,...,...
1812287,@NB501960:812:HH53WAFX5:4:21612:8584:19852 XI:...,>cardiac_neuro_cava_random:REF_CNN3|ENSG000001...,CTTTACTTTTCTCTGGCCAGTCAAAAAAGGAAAAAGAAAAAAAAAA...,270M
1812288,@NB501960:812:HH53WAFX5:4:21612:23901:19994 XI...,>cardiac_neuro_cava_random:REF_AHDC1|ENSG00000...,GTCATCACAGTTATGCACACCAGTCCTCTCCCCCACTGCAAATATG...,270M
1812289,@NB501960:812:HH53WAFX5:4:21612:18136:20141 XI...,>cardiac_neuro_cava_random:REF_CNN3|ENSG000001...,AAGAAAGGACACATTTTCCCAATAGTGACATCACTAAAAGCAATTT...,270M
1812290,@NB501960:812:HH53WAFX5:4:21612:8167:20283 XI:...,>MK:tile_47747|chr5-87945240+87945510|T-A-1_forw,ATATCAAAACTGATTACTAGCTGAGCTTTTGTGTCCATAAAATAAA...,270M


In [123]:
# add oligo_name: without first char (>) and split header at (last) _ but multiple _ in header 
exact_matches_df['pre_oligo_name'] = exact_matches_df['header'].str.split('_').str[0:-1].str.join('_')
exact_matches_df['oligo_name'] = exact_matches_df['pre_oligo_name'].str.lstrip('>')
exact_matches_df
# # print full oligo_name entries of some rows 
# exact_matches_df['oligo_name'].head(10).tolist()[0]
# drop pre_oligo_name column
exact_matches_df = exact_matches_df.drop(['pre_oligo_name'], axis=1)

# only unique oligo_names (drop duplicates)
exact_matches_df_no_dup = exact_matches_df.drop_duplicates(subset=['oligo_name'])
exact_matches_df_no_dup

,read,header,seq,match_length,oligo_name
0,@NB501960:812:HH53WAFX5:1:11101:21622:1110 XI:...,>cardiac_neuro_cava_random:REF_PPP2R2A|ENSG000...,ACTGCAATGCTGCTAATGGATTTTGAGTTGTGGGTACCATATACCC...,270M,cardiac_neuro_cava_random:REF_PPP2R2A|ENSG0000...
1,@NB501960:812:HH53WAFX5:1:11101:4019:1255 XI:Z...,>cardiac_neuro_cava_random:REF_ANK2|ENSG000001...,GTTTTTAGTTGTGTTTAAATCAGGAATTTCCTTACTTTTCAAAATT...,270M,cardiac_neuro_cava_random:REF_ANK2|ENSG0000014...
2,@NB501960:812:HH53WAFX5:1:11101:3848:1256 XI:Z...,>cardiac_neuro_cava_random:REF_LMNA|ENSG000001...,TCTTAAATTATCTGAATCTCTTCTGAAGACAGACCTATTAGCTTTT...,270M,cardiac_neuro_cava_random:REF_LMNA|ENSG0000016...
3,@NB501960:812:HH53WAFX5:1:11101:5659:1395 XI:Z...,>MK:newcore_110746|chr13-80235318+80235587|ref...,GCTTCTAGTCATAGACAATGGCAATTGCAGAGCAAGACTAATAACA...,270M,MK:newcore_110746|chr13-80235318+80235587|refe...
4,@NB501960:812:HH53WAFX5:1:11101:23682:1535 XI:...,>cardiac_neuro_cava_random:ALT_RAI1|ENSG000001...,GAGCAACGCAGGAGAGCAGAAAGGCGAGTGCCCTGGGCTCAGGGGG...,270M,cardiac_neuro_cava_random:ALT_RAI1|ENSG0000010...
...,...,...,...,...,...
1679066,@NB501960:812:HH53WAFX5:4:11412:15223:3491 XI:...,>cardiac_neuro_cava_random:ALT_PIGQ|ENSG000000...,TTTTGGACAAGTGAGTTAATGTCCACCTCCAGTCCTTAGCACTTGC...,270M,cardiac_neuro_cava_random:ALT_PIGQ|ENSG0000000...
1686178,@NB501960:812:HH53WAFX5:4:21411:3048:11026 XI:...,>cardiac_neuro_cava_random:REF_IGLV3-25|ENSG00...,TGCCCAGCGAGACCTGAGTGGTTTTTTTTTTCATTTGTGTGAAATG...,270M,cardiac_neuro_cava_random:REF_IGLV3-25|ENSG000...
1688461,@NB501960:812:HH53WAFX5:4:21511:21490:14960 XI...,>cardiac_neuro_cava_random:REF_RERE|ENSG000001...,GCCCACGCCCTCTGCCACTGCAGTTCCCCCACAGGGCTCCCCCACG...,270M,cardiac_neuro_cava_random:REF_RERE|ENSG0000014...
1752030,@NB501960:812:HH53WAFX5:1:11101:10956:17409 XI...,>C_SLEA:SLEA_hg18:chr2:210861483-210861650|57:...,TAGGCTTCTCAAAAGTTATTTTTAAAGACTGAGGAATTAGGCACCT...,270M,C_SLEA:SLEA_hg18:chr2:210861483-210861650|57:V...


In [ ]:
### with a function: (!TODO )



In [126]:
# bowtie: 
# left join of exact_matches_df_no_dup and missing_bowtie on oligo_name
missing_bowtie = missing_bowtie.drop(['label'], axis=1)
bowtie_missing = missing_bowtie.merge(exact_matches_df_no_dup, on='oligo_name', how='left')
bowtie_missing

# get the oligo_names non nan
# bowtie2_missing[bowtie2_missing['read'].notna()]['oligo_name'].tolist()
bowtie_missing[bowtie_missing['read'].notna()]
print("Number of missing reads with exact match in design: ", len(bowtie_missing[bowtie_missing['read'].notna()]))
# check if bowtie_missing has 0 in matched still
bowtie_missing[bowtie_missing['read'].notna()]['mapped_reads'].value_counts() # no 0 in mapped reads
# Number of missing reads with exact match in design:  66
# No 0 in mapped reads

# bowtie2:
# left join of exact_matches_df_no_dup and missing_bowtie on oligo_name
missing_bowtie2 = missing_bowtie2.drop(['label'], axis=1)
bowtie2_missing = missing_bowtie2.merge(exact_matches_df_no_dup, on='oligo_name', how='left')
bowtie2_missing

# get the oligo_names non nan
# bowtie2_missing[bowtie2_missing['read'].notna()]['oligo_name'].tolist()
bowtie2_missing[bowtie2_missing['read'].notna()]
print("Number of missing reads with exact match in design: ", len(bowtie2_missing[bowtie2_missing['read'].notna()]))
# check if bowtie2_missing has 0 in matched still
bowtie2_missing[bowtie2_missing['read'].notna()]['mapped_reads'].value_counts() # no 0 in mapped reads
# Number of missing reads with exact match in design:  40
# No 0 in mapped reads



Number of missing reads with exact match in design:  66
Number of missing reads with exact match in design:  40


mapped_reads
1    10
2    10
7     6
4     5
9     3
6     3
5     2
8     1
Name: count, dtype: int64

In [110]:
bowtie2_missing

,oligo_name,length,mapped_reads,unmapped_reads,read,header,seq,match_length
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,300,9,0,NaN,NaN,NaN,NaN
1,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,300,7,0,NaN,NaN,NaN,NaN
2,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,300,1,0,NaN,NaN,NaN,NaN
3,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,300,5,0,NaN,NaN,NaN,NaN
4,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,300,8,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2122,MK:tile_985|chr1-33363966+33364235|LC28t6,300,4,0,NaN,NaN,NaN,NaN
2123,MK:tile_985|chr1-33363966+33364235|LC28t7,300,3,0,NaN,NaN,NaN,NaN
2124,MK:tile_985|chr1-33363966+33364235|LC28t9,300,7,0,NaN,NaN,NaN,NaN
2125,MK:tile_30307|chr3-171305106+171305375|scrambl...,300,0,0,NaN,NaN,NaN,NaN


In [109]:
bwa_missing_seqs

,oligo_name,label,barcode,quality,number of matches
1725,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,cardiac_neuro_cava_random,NaN,NaN,NaN
3539,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,cardiac_neuro_cava_random,NaN,NaN,NaN
8201,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,cardiac_neuro_cava_random,NaN,NaN,NaN
13209,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,cardiac_neuro_cava_random,NaN,NaN,NaN
31540,cardiac_neuro_cava_random:RERE|ENSG00000142599...,cardiac_neuro_cava_random,NaN,NaN,NaN
...,...,...,...,...,...
7028722,MK:tile_985|chr1-33363966+33364235|LC28t6,MK,NaN,NaN,NaN
7028723,MK:tile_985|chr1-33363966+33364235|LC28t7,MK,NaN,NaN,NaN
7028725,MK:tile_985|chr1-33363966+33364235|LC28t9,MK,NaN,NaN,NaN
7068818,MK:tile_30307|chr3-171305106+171305375|scrambl...,MK,NaN,NaN,NaN


In [114]:
exact_matches_df

,read,header,seq,match_length,oligo_name
0,@NB501960:812:HH53WAFX5:1:11101:21622:1110 XI:...,>cardiac_neuro_cava_random:REF_PPP2R2A|ENSG000...,ACTGCAATGCTGCTAATGGATTTTGAGTTGTGGGTACCATATACCC...,270M,cardiac_neuro_cava_random:REF_PPP2R2A|ENSG0000...
1,@NB501960:812:HH53WAFX5:1:11101:4019:1255 XI:Z...,>cardiac_neuro_cava_random:REF_ANK2|ENSG000001...,GTTTTTAGTTGTGTTTAAATCAGGAATTTCCTTACTTTTCAAAATT...,270M,cardiac_neuro_cava_random:REF_ANK2|ENSG0000014...
2,@NB501960:812:HH53WAFX5:1:11101:3848:1256 XI:Z...,>cardiac_neuro_cava_random:REF_LMNA|ENSG000001...,TCTTAAATTATCTGAATCTCTTCTGAAGACAGACCTATTAGCTTTT...,270M,cardiac_neuro_cava_random:REF_LMNA|ENSG0000016...
3,@NB501960:812:HH53WAFX5:1:11101:5659:1395 XI:Z...,>MK:newcore_110746|chr13-80235318+80235587|ref...,GCTTCTAGTCATAGACAATGGCAATTGCAGAGCAAGACTAATAACA...,270M,MK:newcore_110746|chr13-80235318+80235587|refe...
4,@NB501960:812:HH53WAFX5:1:11101:23682:1535 XI:...,>cardiac_neuro_cava_random:ALT_RAI1|ENSG000001...,GAGCAACGCAGGAGAGCAGAAAGGCGAGTGCCCTGGGCTCAGGGGG...,270M,cardiac_neuro_cava_random:ALT_RAI1|ENSG0000010...
...,...,...,...,...,...
1812287,@NB501960:812:HH53WAFX5:4:21612:8584:19852 XI:...,>cardiac_neuro_cava_random:REF_CNN3|ENSG000001...,CTTTACTTTTCTCTGGCCAGTCAAAAAAGGAAAAAGAAAAAAAAAA...,270M,cardiac_neuro_cava_random:REF_CNN3|ENSG0000011...
1812288,@NB501960:812:HH53WAFX5:4:21612:23901:19994 XI...,>cardiac_neuro_cava_random:REF_AHDC1|ENSG00000...,GTCATCACAGTTATGCACACCAGTCCTCTCCCCCACTGCAAATATG...,270M,cardiac_neuro_cava_random:REF_AHDC1|ENSG000001...
1812289,@NB501960:812:HH53WAFX5:4:21612:18136:20141 XI...,>cardiac_neuro_cava_random:REF_CNN3|ENSG000001...,AAGAAAGGACACATTTTCCCAATAGTGACATCACTAAAAGCAATTT...,270M,cardiac_neuro_cava_random:REF_CNN3|ENSG0000011...
1812290,@NB501960:812:HH53WAFX5:4:21612:8167:20283 XI:...,>MK:tile_47747|chr5-87945240+87945510|T-A-1_forw,ATATCAAAACTGATTACTAGCTGAGCTTTTGTGTCCATAAAATAAA...,270M,MK:tile_47747|chr5-87945240+87945510|T-A-1


In [117]:
bwa_missing_seqs

,oligo_name,label,barcode,quality,number of matches
1725,cardiac_neuro_cava_random:PRDM16|ENSG000001426...,cardiac_neuro_cava_random,NaN,NaN,NaN
3539,cardiac_neuro_cava_random:SZT2|ENSG00000198198...,cardiac_neuro_cava_random,NaN,NaN,NaN
8201,cardiac_neuro_cava_random:ST3GAL3|ENSG00000126...,cardiac_neuro_cava_random,NaN,NaN,NaN
13209,cardiac_neuro_cava_random:NOS1AP|ENSG000001989...,cardiac_neuro_cava_random,NaN,NaN,NaN
31540,cardiac_neuro_cava_random:RERE|ENSG00000142599...,cardiac_neuro_cava_random,NaN,NaN,NaN
...,...,...,...,...,...
7028722,MK:tile_985|chr1-33363966+33364235|LC28t6,MK,NaN,NaN,NaN
7028723,MK:tile_985|chr1-33363966+33364235|LC28t7,MK,NaN,NaN,NaN
7028725,MK:tile_985|chr1-33363966+33364235|LC28t9,MK,NaN,NaN,NaN
7068818,MK:tile_30307|chr3-171305106+171305375|scrambl...,MK,NaN,NaN,NaN


In [122]:
# how many unique oligo_names in exact_matches_df
exact_matches_df['oligo_name'].value_counts()
# len(exact_matches_df['oligo_name'].unique()) # 2800

oligo_name
GC_GABA_Chengyu:GABA|chr10:26798769-26799038|+|1.64                                                                                                                                          9818
MK:newcore_110746|chr13-80235318+80235587|reference                                                                                                                                          9792
MK:tile_38662|chr7-20964368+20964637|reference                                                                                                                                               9746
GC_GABA_Chengyu:GABA|chr10:26798784-26799053|+|1.64                                                                                                                                          7683
GC_GABA_Chengyu:GABA|chr10:26798844-26799113|+|1.71                                                                                                                                          7503
                   

In [129]:
# bwa:
bwa_missing_mod = bwa_missing_seqs.drop(['barcode', 'quality', 'number of matches'], axis=1)
bwa_missing_mod
bwa_missing = bwa_missing_mod.merge(exact_matches_df_no_dup, on='oligo_name', how='left')
bwa_missing

# get the oligo_names non nan
# bowtie2_missing[bowtie2_missing['read'].notna()]['oligo_name'].tolist()
bwa_missing[bwa_missing['read'].notna()]
print("Number of missing reads with exact match in design: ", len(bwa_missing[bwa_missing['read'].notna()]))
# check if bwa_missing has 0 in matched still
bwa_missing[bwa_missing['read'].notna()]['label'].value_counts() # no 0 in mapped reads
# Number of missing reads with exact match in design:  66
# No 0 in mapped reads


Number of missing reads with exact match in design:  2800


label
cardiac_neuro_cava_random    2639
GC_Mendelian_variants          58
MK                             54
GC_GABA_Chengyu                 9
GC_Glut_Chengyu                 9
GC_Kircher                      3
GC_Atrial_fib                   3
C_negative_neuron_MK            3
C_negative_neuron_NP            3
C_positive_neuron_MK            3
C_positive_neuron_NP            3
C_SLEA                          3
GC_Vista                        3
GC_Selvarajan                   2
GC_Hon                          2
C_positive_heart_AB             2
C_positive_heart_MK             1
Name: count, dtype: int64